# Учебный пример 3 × 2 × 3 — I2-A-04
Все данные синтетические. Три заказа по 1 т смеси; каждому разрешены H и L. Расходы — т/т, запасы — т, q — условные баллы, цель — балл·т. H и L здесь могут выбираться независимо для каждого заказа.


In [1]:
from itertools import product
from copy import deepcopy
def make_case():
    return {
        'synthetic': True, 'period': 'educational',
        'orders': {str(i): {'demand_t': 1., 'candidates': ['H', 'L']} for i in range(1,4)},
        'recipes': {
            'H': {'usage_t_per_t': {'A': .6, 'B': .3, 'C': .1}},
            'L': {'usage_t_per_t': {'A': .2, 'B': .3, 'C': .5}}},
        'stocks': {'A': 1.0, 'B': .9, 'C': 1.1}}

def default_q():
    return {(str(i), r): v for i, pair in enumerate([(.95,.70),(.90,.80),(.85,.75)], 1)
            for r,v in zip(['H','L'],pair)}

def enumerate_all(case, q):
    """Independent arithmetic: does not call solver, validator or scorer."""
    rows = []
    ids = list(case['orders'])
    for choice in product(*(case['orders'][i]['candidates'] for i in ids)):
        used = {c: sum(case['orders'][i]['demand_t'] *
                       case['recipes'][r]['usage_t_per_t'].get(c,0.)
                       for i,r in zip(ids,choice)) for c in case['stocks']}
        obj = sum(case['orders'][i]['demand_t']*q[i,r] for i,r in zip(ids,choice))
        feasible = all(used[c] <= case['stocks'][c]+1e-8 for c in used)
        rows.append({'choices': '/'.join(choice), **used, 'objective':obj,'feasible':feasible})
    return rows

case=make_case(); q=default_q()
print(case)
print(q)

{'synthetic': True, 'period': 'educational', 'orders': {'1': {'demand_t': 1.0, 'candidates': ['H', 'L']}, '2': {'demand_t': 1.0, 'candidates': ['H', 'L']}, '3': {'demand_t': 1.0, 'candidates': ['H', 'L']}}, 'recipes': {'H': {'usage_t_per_t': {'A': 0.6, 'B': 0.3, 'C': 0.1}}, 'L': {'usage_t_per_t': {'A': 0.2, 'B': 0.3, 'C': 0.5}}}, 'stocks': {'A': 1.0, 'B': 0.9, 'C': 1.1}}
{('1', 'H'): 0.95, ('1', 'L'): 0.7, ('2', 'H'): 0.9, ('2', 'L'): 0.8, ('3', 'H'): 0.85, ('3', 'L'): 0.75}


In [2]:
rows=enumerate_all(case,q)
print('Назначение | A, т | B, т | C, т | цель | допустимо')
for row in rows:
 print(row['choices'],*[round(row[k],4) for k in ('A','B','C','objective')],row['feasible'],sep=' | ')
assert len(rows)==8
best=max((v for v in rows if v['feasible']),key=lambda v:v['objective'])
assert best['choices']=='H/L/L' and abs(best['objective']-2.5)<1e-8
print('Оптимум перебора:',best)

Назначение | A, т | B, т | C, т | цель | допустимо
H/H/H | 1.8 | 0.9 | 0.3 | 2.7 | False
H/H/L | 1.4 | 0.9 | 0.7 | 2.6 | False
H/L/H | 1.4 | 0.9 | 0.7 | 2.6 | False
H/L/L | 1.0 | 0.9 | 1.1 | 2.5 | True
L/H/H | 1.4 | 0.9 | 0.7 | 2.45 | False
L/H/L | 1.0 | 0.9 | 1.1 | 2.35 | True
L/L/H | 1.0 | 0.9 | 1.1 | 2.35 | True
L/L/L | 0.6 | 0.9 | 1.5 | 2.25 | False
Оптимум перебора: {'choices': 'H/L/L', 'A': 1.0, 'B': 0.8999999999999999, 'C': 1.1, 'objective': 2.5, 'feasible': True}


Вручную H/L/L: A=0.6+0.2+0.2=1.0 т; B=0.3+0.3+0.3=0.9 т; C=0.1+0.5+0.5=1.1 т. Цель=0.95+0.80+0.75=2.50 балл·т. Два H превышают запас A; все L превышают запас C. Допустимы только варианты с одним H. Перебор использует собственную арифметику, без вызова MILP, валидатора и функции оценки.

## I2-A-05 — независимая сборка MILP и три сценария
Ожидания: база — H/L/L, цель 2.50; запас B=0.8 — допустимого плана нет; q(1,H)=0.1 — оптимум 2.35, два равных варианта L/H/L и L/L/H. Для прежнего H/L/L новая цель равна 1.65. Нормы и потребность не меняются.

In [3]:
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp
def solve(case,q):
    pairs = [(o,r) for o,v in case['orders'].items() for r in v['candidates']]
    matrix=[]; lb=[]; ub=[]
    for o in case['orders']:
        matrix.append([float(oo==o) for oo,r in pairs]);lb.append(1.);ub.append(1.)
    for component, stock in case['stocks'].items():
        matrix.append([case['orders'][o]['demand_t'] *
                       case['recipes'][r]['usage_t_per_t'].get(component,0.) for o,r in pairs])
        lb.append(-np.inf);ub.append(stock)
    result=milp(c=-np.array([case['orders'][o]['demand_t']*q[o,r] for o,r in pairs]),
                integrality=np.ones(len(pairs)),bounds=Bounds(0,1),
                constraints=LinearConstraint(np.array(matrix),lb,ub))
    plan=[] if result.x is None else [{'order_id':o,'recipe_id':r}
                                     for (o,r),x in zip(pairs,result.x) if x>.5]
    return result,plan

for name in ('base','deficit','changed_q'):
 c=deepcopy(case); qq=dict(q)
 if name=='deficit':c['stocks']['B']=0.8
 if name=='changed_q':qq['1','H']=0.1
 rr=enumerate_all(c,qq); feasible=[v for v in rr if v['feasible']]
 result,plan=solve(c,qq)
 expected=None if not feasible else max(v['objective'] for v in feasible)
 print(name,'перебор:',expected,'MILP статус:',result.status,'цель:',None if result.fun is None else -result.fun,'план:',plan)
 if expected is None:assert result.status==2 and not plan
 else:assert result.success and abs(-result.fun-expected)<1e-6
print('Смена q для прежнего H/L/L:',0.1+0.8+0.75)
assert abs((0.1+0.8+0.75)-1.65)<1e-8

base перебор: 2.5 MILP статус: 0 цель: 2.5 план: [{'order_id': '1', 'recipe_id': 'H'}, {'order_id': '2', 'recipe_id': 'L'}, {'order_id': '3', 'recipe_id': 'L'}]
deficit перебор: None MILP статус: 2 цель: None план: []
changed_q перебор: 2.35 MILP статус: 0 цель: 2.35 план: [{'order_id': '1', 'recipe_id': 'L'}, {'order_id': '2', 'recipe_id': 'L'}, {'order_id': '3', 'recipe_id': 'H'}]
Смена q для прежнего H/L/L: 1.65
